# DAPT deneyi — kalp modelini önce bitkiye tanıtmak (HuBERT-ECG, 6 saatlik dilim)

**1)** **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU → Kaydet**

**2)** **Çalışma zamanı → Tümünü çalıştır** (uyarı çıkarsa *Yine de çalıştır*), Drive iznini ver.

| Adım | İş | Süre (T4, tahmini) |
|---|---|---|
| 4 | Duman testi | < 2 dk |
| 5 | DAPT: kalp modeli ve rastgele model etiketsiz bitki verisiyle uyarlanır | ~20–30 dk |
| 6 | Donmuş sonda: 4 kol | ~2 dk |
| 7 | İnce ayar: 4 kol × 12 bitki | ~15–25 dk |
| 8 | Özet | < 1 dk |

Her adım Drive'a (`bitkiEkg_sonuclar/dapt/`) yazar; oturum koparsa **Tümünü çalıştır** yeter, biten kısımlar atlanır. Ayarlar deney öncesi sabittir; değiştirme.

**Sızıntı yok:** DAPT verisi yalnızca kontrol domates bitkileri + sarmaşık; test edilen 12 bitki hiç girmez.

## 1. GPU kontrolü

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU yok! Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU, sonra Tümünü çalıştır.'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)

## 2. Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
KOK = Path('/content/drive/MyDrive/bitkiEkg_sonuclar/dapt')
KOK.mkdir(parents=True, exist_ok=True)
print('Sonuç klasörü:', KOK)

## 3. Kod ve veri

In [ ]:
!rm -rf /content/bitkiEkg
!git clone -q -b onDeneme --depth 1 https://github.com/melihakcam/bitkiEkg.git /content/bitkiEkg
!pip install -q "transformers>=4.40,<5" safetensors huggingface_hub
import sys
sys.path.insert(0, '/content/bitkiEkg/src')
import transformers
assert int(transformers.__version__.split('.')[0]) < 5, 'transformers 5 yüklü: Çalışma zamanı → Yeniden başlat, sonra Tümünü çalıştır.'
import numpy as np, pandas as pd, json
from bitki_ekg.dapt import DAPT_AYARLARI, dapt_egit, hubert_govde
from bitki_ekg.ince_ayar import AYARLAR, Siniflandirici, lopo_calistir
from bitki_ekg.sonda import hubert_gomme, sonda_degerlendir

z = np.load('/content/bitkiEkg/data/colab/domates_ikili_6h_500.npz')
X = np.tile(z['L'].astype(np.float32), (1, 12))          # (276, 6000)
y, g = z['y'].astype(int), z['bitki'].astype(int)
d = np.load('/content/bitkiEkg/data/colab/dapt_6h_500.npz')
D = np.tile(d['D'].astype(np.float32), (1, 12))          # (4400, 6000)
assert not set(np.unique(g)) & {4, 5, 6, 7}, 'değerlendirme setinde kontrol bitkisi olmamalı'
print('Değerlendirme:', X.shape, np.bincount(y), '| DAPT:', D.shape)
print('İnce ayar:', AYARLAR)
print('DAPT:', DAPT_AYARLARI)

KOLLAR = {  # ad: (önceden eğitilmiş mi, DAPT var mı)
    'ekg': (True, False), 'rastgele': (False, False),
    'ekg_dapt': (True, True), 'rastgele_dapt': (False, True),
}
def agirlik(ad):
    onceden, dapt = KOLLAR[ad]
    return (KOK / 'govde' / ('onceden' if onceden else 'rastgele') / 'govde.pt') if dapt else None

## 4. Duman testi

In [ ]:
m = Siniflandirici(True, AYARLAR).cuda().eval()
with torch.no_grad():
    c = m(torch.from_numpy(X[:4]).cuda())
assert c.shape == (4, 2) and torch.isfinite(c).all()
del m; torch.cuda.empty_cache()
from bitki_ekg.dapt import DaptAyarlari
import tempfile
with tempfile.TemporaryDirectory() as t:
    p = dapt_egit(D, True, Path(t), 'cuda', DaptAyarlari(epoch=1), sinir=64)
    m = Siniflandirici(True, AYARLAR, govde_agirlik=p).cuda().eval()
    with torch.no_grad():
        assert torch.isfinite(m(torch.from_numpy(X[:4]).cuda())).all()
    del m; torch.cuda.empty_cache()
print('Duman testi geçti.')

## 5. DAPT (etiketsiz bitki verisiyle uyarlama)

In [ ]:
for onceden in (True, False):
    ad = 'onceden' if onceden else 'rastgele'
    print(f'=== DAPT başlangıç: {ad} ===', flush=True)
    dapt_egit(D, onceden, KOK / 'govde' / ad, 'cuda')
    torch.cuda.empty_cache()

## 6. Donmuş sonda (4 kol)

In [ ]:
SONDA = KOK / 'sonda_6h.csv'
if SONDA.exists():
    print('Önceden tamamlanmış, atlandı')
else:
    parcalar = []
    for ad, (onceden, _) in KOLLAR.items():
        govde = hubert_govde(onceden, AYARLAR.tohum)
        if agirlik(ad) is not None:
            govde.load_state_dict(torch.load(agirlik(ad), map_location='cpu'))
        E = hubert_gomme(govde, X, 'cuda')
        parcalar.append(sonda_degerlendir(E, y, g).assign(kol=ad))
        del govde; torch.cuda.empty_cache()
    pd.concat(parcalar).to_csv(SONDA, index=False)
s = pd.read_csv(SONDA)
print((s[s.bolme == 'lopo'].groupby('kol')[['dogruluk', 'auc']].mean() * 100).round(1))

## 7. İnce ayar (4 kol × 12 bitki, LOPO)

In [ ]:
for ad, (onceden, _) in KOLLAR.items():
    print(f'=== İnce ayar: {ad} ===', flush=True)
    lopo_calistir(X, y, g, onceden, KOK / '6h' / ad, 'cuda', govde_agirlik=agirlik(ad))
    torch.cuda.empty_cache()

## 8. Özet (bitince bana "bitti" yazman yeterli)

In [ ]:
satir = []
for ad in KOLLAR:
    for f in sorted((KOK / '6h' / ad).glob('bitki_*.json')):
        r = json.loads(f.read_text(encoding='utf-8'))
        satir.append({'kol': ad, 'bitki': r['test_bitkisi'], 'dogruluk': r['dogruluk'], 'f1': r['f1'],
                      'auc': r['auc'], 'en_iyi_epoch': r['en_iyi_epoch'], 'n_test': r['n_test']})
df = pd.DataFrame(satir)
df.to_csv(KOK / 'ozet_ince_ayar_6h.csv', index=False)
print('Tamamlanan:', df.groupby('kol').size().to_dict())
print((df.groupby('kol')[['dogruluk', 'f1', 'auc']].agg(['mean', 'std']) * 100).round(1))
print('Karşılaştırma (6 sa, LOPO doğruluk): LightGBM 81,2 · MiniRocket 76,1 · HuBERT donmuş 77,9')